# Import des modules

In [ ]:
pip install pandas

In [ ]:
pip install matplotlib

In [ ]:

pip install seaborn

In [59]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
#Selection
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV, 
    cross_validate,
)
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score
from sklearn.inspection import permutation_importance

#Preprocess
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

#Modèles
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

# Chargement des données

In [ ]:
# dataset = pd.read_csv('/Users/ingrid/Documents/OpenClassrooms/PROJETS/dataset.csv')

In [66]:
dataset = pd.read_csv('https://raw.githubusercontent.com/IngridFi70/Projet06/main/data/dataset.csv')

# Modélisation 

### Préparation des features pour la modélisation

Split du dataset en 2 : un avec les outliers, l'autre sans les outliers

In [67]:
# Séparer les outliers et les non-outliers
df_outliers = dataset[dataset['outlier'] == True]
df_clean = dataset[dataset['outlier'] == False]

Séparer les features et la target

In [68]:
# Séparer les features et la target
X_clean = df_clean.drop(columns=['SiteEnergyUse(kBtu)', 'outlier'])
y_clean = df_clean['SiteEnergyUse(kBtu)']
X_outliers = df_outliers.drop(columns=['SiteEnergyUse(kBtu)', 'outlier'])
y_outliers = df_outliers['SiteEnergyUse(kBtu)']

Séparer les données train des données test

In [69]:
# Faire un split sur les non-outliers pour le train/test
X_train, X_clean_test, y_train, y_clean_test = train_test_split(
X_clean, y_clean, test_size=0.2, random_state=42
)

# Ajouter les outliers au jeu de test
X_test = pd.concat([X_clean_test, X_outliers])
y_test = pd.concat([y_clean_test, y_outliers])

# Réinitialiser les index
X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

In [70]:
X_train.columns

Index(['Neighborhood', 'NumberofFloors', 'PropertyGFABuilding(s)',
       'LargestPropertyUseType', 'LargestPropertyUseTypeGFA', 'ratio_steam',
       'ratio_electricity', 'ratio_gas', 'flag_datacenter', 'flag_laboratory',
       'flag_office', 'flag_downtown', 'flag_greatduw', 'flag_lakeu'],
      dtype='object')

## Régression Linéaire

In [ ]:
# model
model = LinearRegression()

# cross val
scores = cross_validate(model, X_train, y_train, cv=5, scoring='r2', return_train_score=True)

# print score du cross val
print(f"Model: {model}\n")
print(" Validation croisée :")
print(f"  Mean train R2: {np.mean(scores['train_score']):.3f}")
print(f"  Mean test R2:  {np.mean(scores['test_score']):.3f}\n")

# fit
model.fit(X_train, y_train)

# predict
y_pred_test = model.predict(X_test)

# print score du predict
print(" Prédiction vs réel :")
print(f"  MSE: {mean_squared_error(y_test, y_pred_test):.3f}")
print(f"  MAE: {mean_absolute_error(y_test, y_pred_test):.3f}")
print(f"  MAPE: {mean_absolute_percentage_error(y_test, y_pred_test):.3f}")
print(f"  R2: {r2_score(y_test, y_pred_test):.3f}")

Model: LinearRegression()

 Validation croisée :
  Mean train R2: 0.357
  Mean test R2:  0.334

 Prédiction vs réel :
  MSE: 115556857930462.453
  MAE: 5818174.382
  MAPE: 1.050
  R2: 0.479


## Création de fonction

In [ ]:
def run_model(model):
    scores = cross_validate(model, X_train, y_train, cv=5, scoring='r2', return_train_score=True) # cross val
    print(f"Modèle: {model}\n")
    print(" Validation croisée :")
    print(f"  Mean train R2: {np.mean(scores['train_score']):.3f}")
    print(f"  Mean test R2:  {np.mean(scores['test_score']):.3f}\n") # print score du cross val
    model.fit(X_train, y_train) # fit
    y_pred_test = model.predict(X_test) # predict
    print(" Prédiction vs réel :")
    print(f"  MSE: {mean_squared_error(y_test, y_pred_test):.3f}")
    print(f"  MAE: {mean_absolute_error(y_test, y_pred_test):.3f}")
    print(f"  MAPE: {mean_absolute_percentage_error(y_test, y_pred_test):.3f}")
    print(f"  R2: {r2_score(y_test, y_pred_test):.3f}") # print score du predict
  



In [99]:
run_model(LinearRegression())

Model: LinearRegression()

 Validation croisée :
  Mean train R2: 0.357
  Mean test R2:  0.334

 Prédiction vs réel :
  MSE: 115556857930462.453
  MAE: 5818174.382
  MAPE: 1.050
  R2: 0.479


## Tests sur plusieurs modèles

In [ ]:
models = {
    'reg': LinearRegression(),
    'ridge': Ridge(),
    'rfr': RandomForestRegressor(),
    'svr': SVR(),
    'dtr': DecisionTreeRegressor()
}

In [ ]:
for name, model in models.items():
    run_model(model)
    print("-----------")

Model: LinearRegression()

 Validation croisée :
  Mean train R2: 0.357
  Mean test R2:  0.334

 Prédiction vs réel :
  MSE: 115556857930462.453
  MAE: 5818174.382
  MAPE: 1.050
  R2: 0.479
-----------
Model: Ridge()

 Validation croisée :
  Mean train R2: 0.355
  Mean test R2:  0.334

 Prédiction vs réel :
  MSE: 116422989167327.578
  MAE: 5835276.946
  MAPE: 1.054
  R2: 0.476
-----------
Model: RandomForestRegressor()

 Validation croisée :
  Mean train R2: 0.924
  Mean test R2:  0.482

 Prédiction vs réel :
  MSE: 232862546020990.719
  MAE: 8228223.981
  MAPE: 1.330
  R2: -0.049
-----------
Model: SVR()

 Validation croisée :
  Mean train R2: -0.130
  Mean test R2:  -0.130

 Prédiction vs réel :
  MSE: 339994493294112.188
  MAE: 11209034.571
  MAPE: 1.311
  R2: -0.531
-----------
Model: DecisionTreeRegressor()

 Validation croisée :
  Mean train R2: 1.000
  Mean test R2:  0.056

 Prédiction vs réel :
  MSE: 241360176905241.031
  MAE: 8468052.624
  MAPE: 1.259
  R2: -0.087
----------

On constate qu'aucun modèle n'est satisfaisant, du moins pas avec les paramètres par défaut des différents modèles.

## Régression Linéaire

In [77]:
reg = LinearRegression()
reg.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [78]:
y_pred_test_reg = reg.predict(X_test)

In [79]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score
print(f"RMSE: {mean_squared_error(y_test, y_pred_test_reg)}")
print(f"MAE: {mean_absolute_error(y_test, y_pred_test_reg)}")
print(f"MAPE: {mean_absolute_percentage_error(y_test, y_pred_test_reg)}")
print(f"R2: {r2_score(y_test, y_pred_test_reg)}")
print(f"score: {reg.score(X_test, y_test)}")

RMSE: 115556857930462.45
MAE: 5818174.382200218
MAPE: 1.0502953139831828
R2: 0.4794788551001091
score: 0.4794788551001091


In [80]:
print(reg.coef_)

[-1.60984177e+04  9.87906474e+04  2.99698689e+01  5.94662653e+04
  1.17256040e+01 -8.45004365e+05 -2.08237305e+06 -9.99603826e+05
  1.44354999e-08  6.42304707e+06  4.17698774e+05 -9.94615020e+04
 -4.62693800e+05  3.35828369e+05]


## SVR

In [83]:
svr=SVR()
svr.fit(X_train, y_train)

,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,tol,0.001
,C,1.0
,epsilon,0.1
,shrinking,True
,cache_size,200
,verbose,False
,max_iter,-1


In [84]:
y_pred_test_svr = svr.predict(X_test)

In [85]:
print(f"RMSE: {mean_squared_error(y_test, y_pred_test_svr)}")
print(f"MAE: {mean_absolute_error(y_test, y_pred_test_svr)}")
print(f"MAPE: {mean_absolute_percentage_error(y_test, y_pred_test_svr)}")
print(f"R2: {r2_score(y_test, y_pred_test_svr)}")
print(f"score: {svr.score(X_test, y_test)}")

RMSE: 339994493294112.2
MAE: 11209034.570925709
MAPE: 1.3108538845132325
R2: -0.5314913028840373
score: -0.5314913028840373


## Random Forest Regressor

In [87]:
rfr=RandomForestRegressor(n_estimators=100, random_state=42)
rfr.fit(X_train, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [88]:
y_pred_test_rfr = rfr.predict(X_test)

In [89]:
print(f"RMSE: {mean_squared_error(y_test, y_pred_test_rfr)}")
print(f"MAE: {mean_absolute_error(y_test, y_pred_test_rfr)}")
print(f"MAPE: {mean_absolute_percentage_error(y_test, y_pred_test_rfr)}")
print(f"R2: {r2_score(y_test, y_pred_test_rfr)}")
print(f"score: {rfr.score(X_test, y_test)}")

RMSE: 232387711708890.12
MAE: 8218276.882301305
MAPE: 1.2759286901601399
R2: -0.046780952041529655
score: -0.046780952041529655


## Decision Tree Regressor

In [90]:
dtr=DecisionTreeRegressor(random_state=42)
dtr.fit(X_train, y_train)

,criterion,'squared_error'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [91]:
y_pred_test_dtr = dtr.predict(X_test)

In [92]:
print(f"RMSE: {mean_squared_error(y_test, y_pred_test_dtr)}")
print(f"MAE: {mean_absolute_error(y_test, y_pred_test_dtr)}")
print(f"MAPE: {mean_absolute_percentage_error(y_test, y_pred_test_dtr)}")
print(f"R2: {r2_score(y_test, y_pred_test_dtr)}")
print(f"score: {dtr.score(X_test, y_test)}")

RMSE: 234903499215607.6
MAE: 8450839.51354346
MAPE: 1.403231570241456
R2: -0.058113214070577124
score: -0.058113214070577124


# SUITE

### Comparaison de différents modèles supervisés

In [ ]:
# CODE COMPARAISON DES MODELES

### Optimisation et interprétation du modèle

In [ ]:
# CODE OPTIMISATION ET INTERPRETATION DU MODELE